# Electrical Block Wiring Diagram
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

**Purpose.** Generate the propulsion/power/avionics wiring diagram from
the converged sizing result and `config/*.yaml` — there is no
hand-maintained drawing file. Re-run this notebook after any design
change (cell count, EDF diameter, hotel load, battery chemistry, ...)
and the diagram regenerates with matching numbers.

The diagram **topology** (which box connects to which) is a fixed
design decision, defined in `src/conceptual_design/electrical_diagram.py`.
Only the **labels** — pack voltage/capacity, operating current, wire
gauge, connector, servo torque — are computed here.

### Outputs
- `out/wiring_diagram.svg` — the diagram
- `out/electrical.yaml` — the computed operating point (voltage, current, gauge, connector)

---

---
## 0 — Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # bare-checkout fallback

from IPython.display import SVG, display

import conceptual_design as cd
from conceptual_design.notebook import nb_setup
from conceptual_design.design_point import load_handoff, solve_design_point

nb = nb_setup(style=False)   # no matplotlib in this notebook
REPO_ROOT, CONFIG_PATH, OUT_PATH, FIG_DIR = (
    nb.repo_root, nb.config, nb.out, nb.fig_dir)

print("conceptual_design", cd.__version__)

---
## 1 — Re-run the sizing loop

Same pattern as NB2–NB7: reconstruct the converged design point from
`config/` so the wiring diagram always matches the current design,
never a stale snapshot.

In [ ]:
# -- Re-run the sizing loop from config/ ---------------------------------------
inp, result = solve_design_point(CONFIG_PATH)

print(f"Converged MTOW : {result.m_total_kg:.3f} kg")
print(f"P_hover (elec) : {result.P_hover_W:.1f} W")
print(f"P_cruise (elec): {result.P_cruise_W:.1f} W")
print(f"Battery mass   : {result.m_battery_kg:.4f} kg")

---
## 2 — Electrical operating point

The mass-closure loop is chemistry/voltage-agnostic (it only tracks Wh
and kg). Cell count (`config/electrical.yaml`) is the one genuinely
free electrical variable; everything else — pack capacity, operating
current, ESC rating, wire gauge, connector — follows from it plus the
converged sizing result.

In [ ]:
from conceptual_design.electrical_diagram import (
    ElectricalParams, compute_operating_point, render_wiring_svg,
    write_wiring_diagram,
)

elec = ElectricalParams.from_yaml(CONFIG_PATH / "electrical.yaml")
op = compute_operating_point(result, inp.batt, elec)

S = elec.battery_series_cells
print(f"Pack           : {S}S, {op.pack_voltage_v:.1f} V nominal, {op.pack_capacity_ah:.2f} Ah")
print(f"Hover current  : {op.hover_current_a:.1f} A")
print(f"Cruise current : {op.cruise_current_a:.1f} A")
print(f"ESC rating     : ≥{op.esc_rating_a:.0f} A cont. ({elec.esc_current_margin:.1f}x hover margin)")
print(f"Main bus       : {op.main_gauge_awg} AWG, {op.main_connector}")
print(f"BEC1 (sensor)  : {elec.bec1_budget_a:.0f} A budget, {op.bec1_gauge_awg} AWG")
print(f"BEC2 (servo/CC): {elec.bec2_budget_a:.0f} A budget, {op.bec2_gauge_awg} AWG")
print()
print(f"Peak mission C-rate (from mass closure): {result.C_rate_peak:.1f} C "
      f"(pack limit {inp.batt.c_rate_max:.0f} C)")

---
## 3 — Render the diagram

In [ ]:
vanes = load_handoff(OUT_PATH, "control_vanes")
ail   = load_handoff(OUT_PATH, "aileron")

svg = render_wiring_svg(
    op, elec, inp.rotor, inp.batt,
    servo_torque_gcm = vanes["servo_torque_req_gcm"],
    aileron_servo_torque_gcm = ail["servo_torque_req_gcm"],
    package_version  = cd.__version__,
)

display(SVG(svg))

---
## 4 — Export

In [ ]:
paths = write_wiring_diagram(svg, op, elec, OUT_PATH)
for k, v in paths.items():
    print(f"{k:<5}: {v}")

---
## Open items before this becomes a build drawing

| Item | Status |
|---|---|
| EDF motor / ESC part numbers | Open — no thrust-stand map yet for a specific 195 mm COTS motor/prop combo (2026-07 design review) |
| Flight controller / companion computer models | Open — "Pixhawk-class" / "companion computer" are placeholders matching `config/avionics.yaml`'s 15 W hotel-load budget |
| Single vs. redundant FC power path | Open — this diagram shows one BEC per rail; a flight-critical build may want diode-ORed redundancy |
| Battery cell count | Fixed at 6S in `config/electrical.yaml` — keeps hover current at a buildable 34–35 A on standard hobby components |

Grounds/returns are common and intentionally not drawn individually —
this is a block diagram, not a full schematic. Crossing lines without a
junction dot are **not** connected.